# PART 3.2: Word2Vec

In [1]:
import os, sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from collections import defaultdict, Counter
from dotenv import load_dotenv

from myapp.search import load_corpus as lc
from project_progress.part_1.data_prep import (
    corpus_df_loading,
    build_terms,
    join_build_terms,
)
from project_progress.part_2.index_tf_idf import create_index_tf_idf

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import preprocess_string


load_dotenv()  # take environment variables from .env

[nltk_data] Downloading package punkt to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Functions to store the time-consuming processing in files to load faster after:

In [15]:
products_filepath = "../../data/products.json"
products_numeric_data_filepath = "../../data/products_numeric_data.json"


def dump_data(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_data(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def get_or_create(filepath, compute_data_function):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        data = compute_data_function()
        with open(filepath, "w+", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return data

In [3]:
# def process_products(corpus):
#     """
#     Function that loads the products of the corpus as a dictionary of pid -> list (of categorical data as a list of processed tokens)

#     :param corpus: Corpus with all the products and all their data.
#     :return products: (dict) pid -> tokens (all preprocessed terms of the categorical data)
#     """
#     products_sentences = {}
#     for product in list(corpus.values()):

#         # Process the documents categorical fields as in the creation of the inverted index, but concatenate all the terms in a single string (in this case we do not care about the fields)
#         subgroups = [
#             [product.title, product.description],
#             [product.brand, product.category, product.sub_category],
#             [
#                 product.seller,
#                 " ".join([detail for detail in product.product_details.values()]),
#             ],
#         ]

#         products_sentences[product.pid] = [
#             join_build_terms(group) for group in subgroups
#         ]

#     return products_sentences

In [16]:
def process_products(corpus):
    """
    Function that loads the products of the corpus as a dictionary of pid -> list (of categorical data as a list of processed tokens)

    :param corpus: Corpus with all the products and all their data.
    :return products: (dict) pid -> tokens (all preprocessed terms of the categorical data)
    """
    products = {}
    for product in list(corpus.values()):
        
        # Process the documents categorical fields as in the creation of the inverted index, but concatenate all the terms in a single string (in this case we do not care about the fields)
        products[product.pid] = join_build_terms([product.title, 
                                            product.description,
                                            product.brand,
                                            product.category,
                                            product.sub_category,
                                            product.seller,
                                            " ".join([detail for detail in product.product_details.values()])])
    return products

Now, we load the corpus:

In [4]:
json_path = "../../data/fashion_products_dataset.json"
corpus = corpus_df_loading(json_path)

In [17]:
# Preprocess the corpus to get the products
product_sentences = get_or_create(products_filepath, lambda: process_products(corpus))

## Document Representation using Word2Vec 

In [18]:
sentence_list = list()
for doc_sen in product_sentences.values():
    sentence_list.extend(doc_sen)

In [19]:
w2v_model = Word2Vec(
    sentences=sentence_list, vector_size=100, window=7, min_count=5, negative=10, sg=1
)

In [8]:
def build_text_terms(sentences):

    term_list = []
    for sentence in sentences:
        term_list.extend(sentence)

    return term_list

In [20]:
def get_embedding(text, model:Word2Vec, preprocess=preprocess_string):

    doc_terms = build_text_terms(text)
    word_embeddings = [model.wv[word] if word in model.wv else np.zeros(model.vector_size) for word in doc_terms]

    embedding = np.mean(word_embeddings, axis=0)
    return embedding

In [10]:
queries = [
    "western leather jacket men",  # context, material, specific cloth, gender
    "cotton innerwear man",  # material, specific cloth, gender
    "yellow black t-shirt women xl",  # adjectives, specific cloth, gender, size
    "casual comfortable blue trousers women",  # context, adjectives, specific cloth, gender
    "breathable sports clothes winter",
]  # adjectives, context, general clothes, context

In [21]:
doc2vec = {}

for pid, text in product_sentences.items():
    doc2vec[pid] = get_embedding(text, w2v_model)

In [12]:
def cosine_similarity(doc_rep, query_rep):
    dot_prod = np.dot(doc_rep, query_rep)
    similarity = dot_prod / np.linalg.norm(doc_rep)

    return similarity

In [13]:
def get_ranking(query, doc2vec, w2v_model, preprocess=preprocess_string):
    query_terms = preprocess_string(query)
    query_embedding = get_embedding(query_terms, w2v_model)

    sim_scores = []
    for pid, doc_embedding in doc2vec.items():
        score = cosine_similarity(doc_embedding,query_embedding)
        sim_scores.append([score, pid])
    
    sim_scores.sort(key = lambda x: x[0], reverse=True)

    return sim_scores

In [22]:
get_ranking(queries[0], doc2vec, w2v_model)


[[np.float32(1.1201152), 'JCKFB7THY4PFUXHM'],
 [np.float32(1.1185626), 'JCKFWZYYZD7TWXY4'],
 [np.float32(1.1176975), 'JCKFWZYYVFP6EATF'],
 [np.float32(1.1176975), 'JCKFWZYYHZ6DQDEQ'],
 [np.float32(1.1174923), 'JCKFWPYHKAGXGYYK'],
 [np.float32(1.1174923), 'JCKFWPYHFHHAQHGD'],
 [np.float32(1.1173418), 'JCKFWZYYHZ2KVZWZ'],
 [np.float32(1.1173418), 'JCKFWZYYSNREFCFU'],
 [np.float32(1.1168137), 'JCKFWZYYKMPPFNUV'],
 [np.float32(1.1168137), 'JCKFWZYY6HGTZ8KE'],
 [np.float32(1.116784), 'JCKFMV5XJJQMNASZ'],
 [np.float32(1.1167681), 'JCKFWZ8HDMZZVYNC'],
 [np.float32(1.1167139), 'JCKFWZBYPATZVGPB'],
 [np.float32(1.1166518), 'JCKFWZYYPNPGU7GG'],
 [np.float32(1.1166518), 'JCKFWZYYREG8GDHG'],
 [np.float32(1.1166518), 'JCKFWZYY82ARQPQ5'],
 [np.float32(1.1166518), 'JCKFWZYYCBDZ478H'],
 [np.float32(1.1166518), 'JCKFWZYYHMBHWF5C'],
 [np.float32(1.1166327), 'JCKFZSPYYZGCYGZK'],
 [np.float32(1.1165589), 'JCKFWZBYSXCEQDYD'],
 [np.float32(1.1164552), 'JCKFWYNCWHNRQ4BA'],
 [np.float32(1.1163772), 'JCKFWZYYY